<a href="https://colab.research.google.com/github/raz0208/Natural-Language-Processing-Practices/blob/main/TopicModelling/NLP_TopicsModellingPractices.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Topic Modelling Practices in NLP


1. Semantic Signal Separation
2. KeyNMF
3. ClusteringTopicModel



### 1. Semantic Signal Separation

In [ ]:
!pip install -q transformers datasets huggingface_hub

In [ ]:
from huggingface_hub import login

login("")

In [ ]:
# Install turftopic libraries
!pip install turftopic
!pip install datasets

In [ ]:
# import required libraries
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import os
from turftopic import SemanticSignalSeparation
from datasets import load_dataset
from sentence_transformers import SentenceTransformer

In [ ]:
# Read and load dataset
ds = pd.read_csv('political_ideologies_train.csv')
texts = ds["statement"]

print(texts)

0       Climate change, and the escalating environment...
1       I believe in the foundational importance of th...
2       I firmly believe that the principle of separat...
3       I firmly believe in the separation of church a...
4       I firmly believe in the power of free markets ...
                              ...                        
2555    I believe in the power of free markets to driv...
2556    I firmly believe in the traditional family str...
2557    Every individual, regardless of their gender, ...
2558    I firmly believe in the significance of religi...
2559    I firmly believe in the principle of individua...
Name: statement, Length: 2560, dtype: object


In [ ]:
ds

,statement,label,issue_type,__index_level_0__
0,"Climate change, and the escalating environment...",1,1,465
1,I believe in the foundational importance of th...,0,2,1191
2,I firmly believe that the principle of separat...,1,6,2440
3,I firmly believe in the separation of church a...,1,6,2406
4,I firmly believe in the power of free markets ...,0,0,1903
...,...,...,...,...
2555,I believe in the power of free markets to driv...,0,0,1907
2556,I firmly believe in the traditional family str...,0,2,1139
2557,"Every individual, regardless of their gender, ...",1,7,48
2558,I firmly believe in the significance of religi...,1,6,822


In [ ]:
encoder = SentenceTransformer('paraphrase-MiniLM-L12-v2')
embeddings = encoder.encode(texts, show_progress_bar=True)

In [ ]:
print(embeddings)

[[-0.20092632  0.33961082  0.02055947 ... -0.14055775 -0.09066175
  -0.05947793]
 [-0.05251725  0.2111203  -0.2350792  ... -0.19393641  0.06882618
  -0.02158776]
 [-0.06461112  0.07165432 -0.08635489 ... -0.05988945  0.02278206
   0.0677686 ]
 ...
 [ 0.2636646   0.07064892 -0.10288888 ...  0.00702172  0.0945406
  -0.02747649]
 [ 0.2114343   0.13071813 -0.26949868 ... -0.00532549 -0.03988217
  -0.08325328]
 [-0.3067356   0.26985478 -0.18770279 ... -0.16361783  0.10027055
  -0.09515021]]


In [ ]:
model = SemanticSignalSeparation(4, encoder=encoder, random_state=42)
doc_topic_matrix = model.fit_transform(texts, embeddings=embeddings)

In [ ]:
model.print_topics(top_k=10)

┏━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Topic ID ┃ Topic Name         ┃ Highest Ranking                        ┃ Lowest Ranking                         ┃
┡━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│        0 │ Religiosity        │ warming, carbon, environment, solar,   │ wealth, taxation, prosperity,          │
│          │                    │ change, planet, greenhouse,            │ wealthiest, tax, profit, taxes,        │
│          │                    │ environmental, shift, fossil           │ entrepreneurship, fiscal, government   │
├──────────┼────────────────────┼────────────────────────────────────────┼────────────────────────────────────────┤
│        1 │ Economic vs Social │ religious, religion, church,           │ spending, fiscal, budget, economic,    │
│          │                    │ religions, faith, faiths, respectful,  │ economies, economy, growth,            │
│          │                    │ respect, distinct, lgbtq               │ investments, economically, productive  │
├──────────┼────────────────────┼────────────────────────────────────────┼────────────────────────────────────────┤
│        2 │ Environmentalism   │ marriage, structure, traditions,       │ healthcare, medical, patient, health,  │
│          │                    │ relationships, structures, tradition,  │ care, healthy, treated, healthier,     │
│          │                    │ tensions, conflicts, conflict, couples │ treatment, treatments                  │
├──────────┼────────────────────┼────────────────────────────────────────┼────────────────────────────────────────┤
│        3 │ Personality        │ freedoms, liberties, iran, freedom,    │ households, adoption, racial,          │
│          │                    │ faith, religion, protection, secure,   │ household, racism, families,           │
│          │                    │ military, security                     │ ethnicity, mothers, childcare, races   │
└──────────┴────────────────────┴────────────────────────────────────────┴────────────────────────────────────────┘

In [ ]:
model.plot_concept_compass(0, 1)

In [ ]:
model.rename_topics({
    0: "Religiosity",
    1: "Economic vs Social",
    2: "Environmentalism",
    3: "Personality"
})

In [ ]:
model.print_topic_distribution("I am a socialist and I am concerned with the growing inequality in our societies. I'd like to see governments do more to prevent the exploitation of workers.")

┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┓
┃ Topic name         ┃ Score ┃
┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━┩
│ Environmentalism   │  0.09 │
│ Personality        │  0.03 │
│ Religiosity        │ -1.13 │
│ Economic vs Social │ -1.32 │
└────────────────────┴───────┘

In [ ]:
import plotly.express as px

df = pd.DataFrame(doc_topic_matrix, columns=model.topic_names)
df["party"] = ["Liberal" if label == 1 else "Conservative" for label in ds["label"]]

fig = px.scatter_matrix(df, dimensions=model.topic_names, color="party", template="plotly_white", width=1000, height=1100)
fig = fig.update_traces(diagonal_visible=False, showupperhalf=False, marker=dict(opacity=0.6))
fig.show()

### 3. Clustering Analysing
#### Building a Taxonomy of Machine Learning Papers

In [ ]:
!pip install -U datasets
# !pip install datasets
!pip install turftopic[umap-learn]
!pip install turftopic[datamapplot]
#!pip install top2vec[visualization]

In [15]:
# import required libraries
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import os
import torch

# From turftopic import SemanticSignalSeparation
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from turftopic import Top2Vec
from turftopic import BERTopic

In [2]:
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
print("Device name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")
print("-------"*10)
!nvidia-smi

CUDA available: True
GPU count: 1
Device name: Tesla T4
----------------------------------------------------------------------
Fri Jun 13 18:35:37 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8              9W /   70W |       2MiB /  15360MiB |      0%      Default |
|            

In [ ]:
# Read and load dataset from huggingface
ds1 = load_dataset("CShorten/ML-ArXiv-Papers", split="train")

# Subsampling dataset
ds1 = ds1.train_test_split(seed=42, test_size=10000)["test"]

abstracts1 = ds1["abstract"]

In [4]:
# show the number of abstracts
print(f"The number of abstracts: {len(abstracts1)}", "\n")

abstracts1[:5]

The number of abstracts: 10000 



["  In this paper we propose $\\epsilon$-Consistent Mixup ($\\epsilon$mu).\n$\\epsilon$mu is a data-based structural regularization technique that combines\nMixup's linear interpolation with consistency regularization in the Mixup\ndirection, by compelling a simple adaptive tradeoff between the two. This\nlearnable combination of consistency and interpolation induces a more flexible\nstructure on the evolution of the response across the feature space and is\nshown to improve semi-supervised classification accuracy on the SVHN and\nCIFAR10 benchmark datasets, yielding the largest gains in the most challenging\nlow label-availability scenarios. Empirical studies comparing $\\epsilon$mu and\nMixup are presented and provide insight into the mechanisms behind\n$\\epsilon$mu's effectiveness. In particular, $\\epsilon$mu is found to produce\nmore accurate synthetic labels and more confident predictions than Mixup.\n",
 '  Autoencoders have useful applications in high energy physics in anomaly

In [ ]:
# # Using TurfTopic default encoder to extract embedding of the dataset
encoder1 = SentenceTransformer("all-MiniLM-L6-v2")
embeddings1 = encoder1.encode(abstracts1, show_progress_bar=True)

In [7]:
# create a dataframe for 5 first embeddings
df = pd.DataFrame(embeddings1[:])
df

,0,1,2,3,4,5,6,7,8,9,...,374,375,376,377,378,379,380,381,382,383
0,-0.116793,-0.048010,0.013904,0.020555,0.102551,-0.002297,-0.001540,-0.038846,-0.093981,-0.128632,...,0.091226,-0.072464,-0.044301,-0.079422,0.023961,0.064886,0.045012,-0.055690,-0.015406,0.007885
1,-0.113509,-0.063588,0.045744,0.069508,0.023963,0.004733,0.011323,-0.020639,0.009974,-0.052237,...,0.004441,0.046467,-0.005209,-0.106404,0.055330,0.031162,-0.015407,0.052612,-0.079166,0.000951
2,-0.046366,-0.062943,0.076053,0.008464,0.087632,0.004908,0.033890,-0.041043,-0.004651,0.000576,...,0.064989,0.014189,0.044394,-0.055709,0.007850,-0.015040,0.030750,-0.056158,0.009977,0.046114
3,-0.022161,-0.081515,0.074852,-0.008076,0.074276,0.006316,0.020354,0.036162,-0.047043,-0.030080,...,0.005349,-0.080112,-0.053862,-0.055705,-0.007194,0.128456,0.031168,0.002715,-0.018317,0.006816
4,-0.024045,-0.038111,0.033548,-0.041262,0.083008,0.015470,0.010767,-0.093849,-0.008791,-0.086870,...,0.003405,-0.008197,-0.003859,-0.026671,0.040897,0.136462,-0.033441,-0.023917,-0.017840,-0.013066
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,-0.022007,-0.025172,-0.025964,0.005955,0.110572,-0.008803,-0.000628,-0.105988,-0.049180,0.014990,...,0.056070,-0.001756,-0.081871,-0.042868,0.103274,-0.039233,0.051121,0.031668,-0.014564,-0.033605
9996,-0.148666,-0.023369,0.058389,0.028600,-0.004698,0.016038,-0.013862,-0.040023,-0.053083,-0.051324,...,-0.025992,-0.010671,-0.037332,-0.074483,-0.027046,0.029448,0.001782,0.074415,0.054203,0.036414
9997,0.005107,-0.029706,0.023024,0.019745,0.017944,-0.002686,0.021389,-0.052526,-0.008535,-0.053164,...,-0.058315,-0.001612,0.014443,-0.068842,-0.024539,0.051659,-0.013413,-0.011464,-0.118150,-0.031719
9998,-0.015600,-0.075435,-0.010884,-0.048085,-0.022817,-0.014012,0.013586,-0.009622,-0.004461,0.013935,...,0.094178,-0.030854,0.029789,-0.050457,0.061554,0.031880,-0.026690,-0.003210,-0.051809,-0.011401


In [8]:
# # show rows 127, 223, 319
# df.iloc[[127, 223, 319]]

In [9]:
print(embeddings1, embeddings1.shape)

[[-0.1167931  -0.04801036  0.01390377 ... -0.05568971 -0.01540568
   0.00788483]
 [-0.11350863 -0.06358757  0.04574388 ...  0.05261219 -0.07916622
   0.00095109]
 [-0.04636589 -0.06294328  0.07605291 ... -0.0561576   0.00997735
   0.04611383]
 ...
 [ 0.00510686 -0.02970619  0.02302423 ... -0.01146424 -0.11814963
  -0.03171895]
 [-0.01559985 -0.07543531 -0.01088388 ... -0.00320966 -0.05180904
  -0.01140063]
 [-0.00397541 -0.04718209  0.06320886 ... -0.04551153 -0.04912112
  -0.032086  ]] (10000, 384)


### Read saved embedding dataset from root

In [12]:
# # save the embedding in a csv file
# np.savetxt("Cluster_Analysis_Default_DS_Embeddings.csv", embeddings, delimiter=",")

In [ ]:
# # Read and load dataset
# emb_ds1 = pd.read_csv('Cluster_Analysis_Default_DS_Embeddings.csv')

# embeddings = emb_ds1.to_numpy()

# print(embeddings, embeddings.shape)

In [23]:
# from sklearn.preprocessing import StandardScaler

# scaler = StandardScaler()
# embeddings1_Sc = scaler.fit_transform(embeddings1)

# from sklearn.preprocessing import MinMaxScaler

# scaler = MinMaxScaler()
# embeddings_MMc = scaler.fit_transform(embeddings1)

In [9]:
# embeddings_Sc

In [ ]:
!pip install bertopic

from bertopic import BERTopic

In [ ]:
model = BERTopic(encoder=encoder1, random_state=42)
topic_data = model.prepare_topic_data(abstracts1, embeddings=embeddings1)

In [17]:
model.print_topics()

┏━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Topic ID ┃ Highest Ranking                                                                                      ┃
┡━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│       -1 │ data, model, learning, models, neural, based, network, training, image, networks                     │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│        0 │ quantum, classical, circuit, qubit, machine, circuits, computers, hamiltonian, noise, near           │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│        1 │ delivery, problem, routing, graph, combinatorial, traveling, facility, problems, driver, requests    │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│        2 │ players, sports, player, team, game, teams, playing, match, matches, games                           │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│        3 │ voltage, grid, drl, simplex, energy, operation, control, power, policy, marl                         │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│        4 │ financial, trading, portfolio, market, reinforcement, asset, price, profit, agent, rl                │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│        5 │ drl, scheduling, resource, wireless, service, radio, reinforcement, channel, vehicles, 5g            │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│        6 │ fairness, fair, groups, bias, discrimination, unfairness, decision, algorithmic, sensitive, group    │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│        7 │ forgetting, continual, catastrophic, cl, memory, tasks, incremental, lifelong, task, new             │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│        8 │ revenue, price, auction, advertising, ad, auctions, ads, pricing, products, bid                      │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│        9 │ channel, wireless, receiver, modulation, iot, csi, communication, coding, snr, decoding              │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│       10 │ games, equilibrium, game, nash, agents, sum, players, agent, player, regret                          │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│       11 │ localization, indoor, location, positioning, csi, position, smartphone, wireless, meters, wall       │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│       12 │ gnns, attacks, graph, gnn, attack, adversarial, graphs, node, robustness, defense                    │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│       13 │ policy, rl, reinforcement, reward, agent, policies, robot, control, action, planning                 │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│       14 │ agent, agents, marl, team, opponent, reinforcement, multi, centralized, incentive, policies          │
├──────────┼────────────────────────────────────────────

In [18]:
model.reduce_topics(n_reduce_to=25)
print(model.hierarchy.cut(3))

Root: 
├── -1: learning, data, model, based, models, neural, training, network, method, performance
├── 0: quantum, classical, learning, machine, circuit, qubit, noise, algorithms, circuits, algorithm
├── 2: players, sports, game, player, team, teams, playing, match, games, matches
├── 56: seismic, data, velocity, cnn, cnns, wave, subsurface, noise, inversion, imaging
├── 66: physics, particle, detector, particles, energy, high, beam, reconstruction, event, detectors
├── 67: process, event, mining, logs, prediction, discovery, intervention, log, business, models
├── 115: nas, search, architecture, architectures, neural, performance, space, cifar, network, 
│   training
├── 122: mixture, mixtures, gaussians, gaussian, algorithm, distributions, reduction, univariate, 
│   binomial, parameters
├── 123: ot, transport, optimal, sinkhorn, wasserstein, regularized, algorithm, problem, measures, 
│   plan
├── 136: nmf, matrix, nonnegative, factorization, matrices, data, gene, ontology, factori

In [19]:
fig = model.hierarchy.plot_tree()
fig.show()

In [21]:
fig = model.hierarchy[298].plot_tree()
fig.show()

In [26]:
# We will reset the hierarchy, so that we can see all topics at once.
model.reset_topics()
fig = model.plot_clusters_datamapplot(hover_text=ds1["title"])
fig.show()

In [23]:
# create a dataframe for 5 first embeddings
df = pd.DataFrame(embeddings1[:])
df

,0,1,2,3,4,5,6,7,8,9,...,374,375,376,377,378,379,380,381,382,383
0,-0.116793,-0.048010,0.013904,0.020555,0.102551,-0.002297,-0.001540,-0.038846,-0.093981,-0.128632,...,0.091226,-0.072464,-0.044301,-0.079422,0.023961,0.064886,0.045012,-0.055690,-0.015406,0.007885
1,-0.113509,-0.063588,0.045744,0.069508,0.023963,0.004733,0.011323,-0.020639,0.009974,-0.052237,...,0.004441,0.046467,-0.005209,-0.106404,0.055330,0.031162,-0.015407,0.052612,-0.079166,0.000951
2,-0.046366,-0.062943,0.076053,0.008464,0.087632,0.004908,0.033890,-0.041043,-0.004651,0.000576,...,0.064989,0.014189,0.044394,-0.055709,0.007850,-0.015040,0.030750,-0.056158,0.009977,0.046114
3,-0.022161,-0.081515,0.074852,-0.008076,0.074276,0.006316,0.020354,0.036162,-0.047043,-0.030080,...,0.005349,-0.080112,-0.053862,-0.055705,-0.007194,0.128456,0.031168,0.002715,-0.018317,0.006816
4,-0.024045,-0.038111,0.033548,-0.041262,0.083008,0.015470,0.010767,-0.093849,-0.008791,-0.086870,...,0.003405,-0.008197,-0.003859,-0.026671,0.040897,0.136462,-0.033441,-0.023917,-0.017840,-0.013066
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,-0.022007,-0.025172,-0.025964,0.005955,0.110572,-0.008803,-0.000628,-0.105988,-0.049180,0.014990,...,0.056070,-0.001756,-0.081871,-0.042868,0.103274,-0.039233,0.051121,0.031668,-0.014564,-0.033605
9996,-0.148666,-0.023369,0.058389,0.028600,-0.004698,0.016038,-0.013862,-0.040023,-0.053083,-0.051324,...,-0.025992,-0.010671,-0.037332,-0.074483,-0.027046,0.029448,0.001782,0.074415,0.054203,0.036414
9997,0.005107,-0.029706,0.023024,0.019745,0.017944,-0.002686,0.021389,-0.052526,-0.008535,-0.053164,...,-0.058315,-0.001612,0.014443,-0.068842,-0.024539,0.051659,-0.013413,-0.011464,-0.118150,-0.031719
9998,-0.015600,-0.075435,-0.010884,-0.048085,-0.022817,-0.014012,0.013586,-0.009622,-0.004461,0.013935,...,0.094178,-0.030854,0.029789,-0.050457,0.061554,0.031880,-0.026690,-0.003210,-0.051809,-0.011401


In [24]:
# Calculate the Standard deviation for df
std_dev = np.std(df, axis=0)

std_dev

,0
0,0.044472
1,0.044867
2,0.041396
3,0.042194
4,0.048787
...,...
379,0.047914
380,0.044293
381,0.044813
382,0.043120


In [25]:
# Print starndard deviations approximately 0
std_dev[std_dev < 0.0001]

,0
127,0.000000e+00
223,0.000000e+00
319,5.260577e-09


In [39]:
mean = np.mean(df, axis=0)
mean

,0
0,-0.050614
1,-0.051454
2,0.021106
3,0.017747
4,0.040029
...,...
379,0.041593
380,0.007266
381,0.006576
382,-0.039145


In [40]:
print("Minimum std deviation across dimensions:", np.min(std_dev))
print("Minimum mean across dimensions:", np.min(mean))
print("Maximum mean across dimensions:", np.max(mean))

Minimum std deviation across dimensions: 0.05091680958867073
Minimum mean across dimensions: -0.07107391953468323
Maximum mean across dimensions: 0.07737497240304947
